# Домашнее задание 1: CBOW своими руками

**Цель.** Реализовать модель **CBOW** (Continuous Bag of Words) с нуля на
PyTorch: собрать обучающие пары из скользящего окна, написать `Dataset` и модель,
обучить её и убедиться, что эмбеддинги улавливают смысл — близкие по значению
слова оказываются рядом по косинусу.

**Что уже готово (писать не нужно):**

* подготовка данных: чтение корпуса, токенизация, словарь, субсэмплинг, split;
* **проверки близости слов** (в конце ноутбука) — полностью готовое решение.

**Что пишете вы** (задания помечены `✏️`):

| Задание | Что делаем | Что дано |
|---|---|---|
| **1** | окна контекста — пары «контекст → центр» | постановка + схема |
| **2** | `torch.utils.data.Dataset` | сигнатуры методов |
| **3** | модель `CBOW` | **только схема** модели |
| **4** | шаг обучения | сигнатура |
| **5** | цикл обучения с валидацией | сигнатура |

**Как сдавать.** В каждой ячейке с `✏️` — заготовка с пропусками
(`...  # <-- YOUR CODE HERE`) и `assert`-проверки, которые печатают
`Задание N OK ✓`, когда решение верно. Ваша задача — заполнить пропуски и
убедиться, что ноутбук проходит сверху вниз без ошибок.

> **Важно: работаем без negative sampling.** В лекции `softmax` по всему словарю
> заменяли negative sampling'ом ради скорости. Здесь мы его **убираем** и считаем
> честный полный `softmax` по словарю: модель предсказывает центр **среди всех
> слов** сразу, а обучаем `CrossEntropyLoss`. Поэтому словарь намеренно небольшой
> — иначе полный `softmax` был бы очень дорогим.

Обучение на CPU — около 2 минут.

## 0. Импорты, seed, устройство

Пишем **device-agnostic** код: CUDA → MPS → CPU. Фиксируем сиды.

In [ ]:
import os, re, time, random, collections
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ---------- воспроизводимость ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---------- device-agnostic ----------
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (8, 4)

print('numpy', np.__version__, '| torch', torch.__version__, '| device:', device)

---

## 1. Корпус и подготовка данных (готово)

### 1.1. Чтение и токенизация

Читаем `.txt`, режем на предложения. Предложения — это «документы» для CBOW:
контекст не должен перетекать через границу предложения.

Хотите свой корпус — положите любой `.txt` рядом и поменяйте `CORPUS_PATH`.
Строки, начинающиеся с `##`, — заголовки статей, их пропускаем.

In [ ]:
CORPUS_PATH = 'corpus.txt'   # <<< любой ваш .txt (UTF-8)

def find_corpus(path):
    '''Ищем файл рядом с ноутбуком (на случай запуска из другой папки).'''
    for c in [path, os.path.join('fall2026/homework1', path), os.path.join('..', path)]:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f'не найден {path} — положите .txt рядом с ноутбуком')

TOKEN_RE = re.compile(r"[a-z0-9']+")

def tokenize(text):
    '''Слова в нижнем регистре: буквы / цифры / апостроф.'''
    return TOKEN_RE.findall(text.lower())

corpus_path = find_corpus(CORPUS_PATH)
text = open(corpus_path, encoding='utf-8').read()
raw_lines = [l.strip() for l in text.splitlines() if l.strip() and not l.startswith('##')]

try:
    import nltk
    nltk.data.find('tokenizers/punkt')
    from nltk.tokenize import sent_tokenize
except (ImportError, LookupError):
    import nltk
    nltk.download('punkt', quiet=True)
    from nltk.tokenize import sent_tokenize

sentences = []
for line in raw_lines:
    for sent in sent_tokenize(line):
        toks = tokenize(sent)
        if len(toks) >= 2:                 # предложения из 1 слова бесполезны
            sentences.append(toks)

all_tokens = [t for s in sentences for t in s]
print('файл:', corpus_path)
print('предложений:', f'{len(sentences):,}', '| токенов:', f'{len(all_tokens):,}')

### 1.2. Словарь

Оставляем слова с частотой `MIN_COUNT` и не больше `MAX_VOCAB` самых частых.
Спец-токены (`<pad>`, `<unk>`) не нужны: незнакомые слова мы просто **выбрасываем**
— цель не классификация, а сами векторы.

> Словарь небольшой (`MAX_VOCAB`) сознательно: полный `softmax` считается по
> `V` классам, и с ростом `V` обучение дорожает линейно.

In [ ]:
MIN_COUNT = 5        # слово должно встретиться хотя бы столько раз
MAX_VOCAB = 3000     # не больше самого частого «хвоста» (см. ~ про softmax)

freq = collections.Counter(all_tokens)
words = [w for w, c in freq.most_common(MAX_VOCAB) if c >= MIN_COUNT]
word2id = {w: i for i, w in enumerate(words)}
id2word = words
V = len(words)

coverage = sum(freq[w] for w in words) / len(all_tokens)
print(f'размер словаря V = {V:,}')
print(f'покрытие корпуса словарём: {coverage:.1%}')
print()
print('Топ-10 частых слов:')
for w, c in freq.most_common(10):
    print(f'  {w:12s} {c:>8,}')

# корпус в виде списка предложений из id (незнакомые слова выброшены)
corpus_ids = [[word2id[w] for w in s if w in word2id] for s in sentences]
corpus_ids = [ids for ids in corpus_ids if len(ids) >= 2]

### 1.3. Train / validation split

Делим **предложения** (не токены) на обучающую и валидационную части, чтобы честно
мерить loss на невиданных контекстах. Разбиение делаем **до** построения пар.

In [ ]:
rng = random.Random(SEED)
shuffled = corpus_ids[:]
rng.shuffle(shuffled)

n = len(shuffled)
n_train = int(0.9 * n)
train_ids = shuffled[:n_train]
val_ids = shuffled[n_train:]

print('train предложений:', f'{len(train_ids):,}')
print('val   предложений:', f'{len(val_ids):,}')
print()
print('Пример предложения (id):', train_ids[0][:15], '...')

---

# Часть 2. Пары «контекст → центр»

Всё, что дальше, — **ваша работа**. Начнём со схемы, как текст превращается в
обучающие примеры.

```text
   предложение (id слов):   ... a  b  c  d  e  f  g ...
                                    │
                        для центра d, окно WINDOW = 3
                                    │
                  контекст: [a, b, c] + [e, f, g]   (3 слова слева + 3 справа)
                                    │
                          пара: (context, target=d)
```

Для **каждой** позиции, у которой есть полное окно слева и справа, получается
одна пара `(контекст, центр)`. Порядок соседей фиксирован: слева направо, центр
исключён. Контекст — длины `2 * WINDOW`.

### ✏️ Задание 1 — окна контекста

Напишите `make_windows(ids, window)`: по списку id одного предложения верните
список пар `(context, target)` для всех позиций с полным окном.

* идите по позициям `i` от `window` до `len(ids) - window - 1` включительно;
* `context` = `window` слов слева (по порядку) **плюс** `window` слов справа;
* `target` = слово в позиции `i`.

**Напишите решение в ячейке ниже.**

In [ ]:
# ✏️ Задание 1 — составим корпус из окон контекста
WINDOW = 3   # полуокно: по 3 слова влево и вправо

def make_windows(ids, window=WINDOW):
    '''Пары (контекст, центр) для одного предложения.

    context — список длины 2*window: window слов слева + window слов справа.
    target  — id центрального слова (int).
    '''
    pairs = []
    # <-- YOUR CODE HERE
    return pairs


# ---------- проверки (Задание 1) ----------
demo = [10, 20, 30, 40, 50, 60, 70]
w_demo = make_windows(demo, 3)
assert w_demo == [([10, 20, 30, 50, 60, 70], 40)], \
    f'для {demo} ожидалось [([10,20,30,50,60,70], 40)], а получено {w_demo}'
assert make_windows([1, 2, 3], 3) == [], 'нет полного окна — пар быть не должно'

n_pairs_train = sum(len(make_windows(ids, WINDOW)) for ids in train_ids)
print('Задание 1 OK ✓ | пар в train:', f'{n_pairs_train:,}')

### ✏️ Задание 2 — `Dataset`

Теперь оберните пары в `torch.utils.data.Dataset`. Он должен собрать **все** пары
по корпусу (одна большая выборка) и по индексу возвращать тензоры.

* в `__init__` соберите `self.samples` — плоский список всех пар по всем
  предложениям (используйте `make_windows`);
* `__len__` — число пар;
* `__getitem__(idx)` — верните `(context, target)`, где `context` —
  тензор `torch.long` формы `(2 * WINDOW,)`, а `target` — **скаляр**-тензор
  `torch.long` (`dim() == 0`).

**Напишите решение в ячейке ниже.**

In [ ]:
# ✏️ Задание 2 — Dataset для CBOW
class CBOWDataset(Dataset):
    '''Возвращает пары (context_ids, target_id) для всех позиций корпуса.'''
    def __init__(self, corpus_ids, window=WINDOW):
        self.window = window
        self.samples = ...        # <-- YOUR CODE HERE: плоский список всех пар

    def __len__(self):
        return ...                # <-- YOUR CODE HERE

    def __getitem__(self, idx):
        ctx, tgt = self.samples[idx]
        # <-- YOUR CODE HERE: вернуть (torch.long тензор контекста, скаляр-тензор центра)
        ...


# ---------- проверки (Задание 2) ----------
train_ds = CBOWDataset(train_ids, WINDOW)
val_ds   = CBOWDataset(val_ids, WINDOW)

ctx0, tgt0 = train_ds[0]
assert len(train_ds) == sum(len(make_windows(s, WINDOW)) for s in train_ids), 'неверное число пар'
assert ctx0.shape == (2 * WINDOW,), f'контекст должен быть длины {2 * WINDOW}, а не {tuple(ctx0.shape)}'
assert ctx0.dtype == torch.long and tgt0.dtype == torch.long, 'нужен torch.long'
assert tgt0.dim() == 0, 'target — скаляр-тензор (dim == 0)'
assert all(0 <= x < V for x in ctx0.tolist() + [tgt0.item()]), 'индексы вне словаря'
print('Задание 2 OK ✓ | train пар =', f'{len(train_ds):,}', '| val пар =', f'{len(val_ds):,}')
print('пример: ctx', ctx0.tolist(), '->', id2word[tgt0.item()])

### `DataLoader` (готово)

Оборачиваем датасеты в `DataLoader`: перемешивание каждый эпоху, батчи,
`num_workers` (параллельная загрузка) и `pin_memory` под CUDA.

> На Windows/Colab `num_workers > 0` иногда ломается при запуске из ноутбука —
> там поставьте `NUM_WORKERS = 0`.

In [ ]:
BATCH_SIZE   = 512
NUM_WORKERS  = 2          # на Windows/Colab поставьте 0 при проблемах
PIN_MEMORY   = (device.type == 'cuda')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=1024, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

ctx_batch, tgt_batch = next(iter(train_loader))
print('батч контекстов:', tuple(ctx_batch.shape), '| батч центров:', tuple(tgt_batch.shape))

---

# Часть 3. Модель CBOW

Классический CBOW предсказывает центр **полным `softmax` по всему словарю**.
Никакого negative sampling (в отличие от лекции) — учим «честный» $V$-классовый
классификатор.

### Схема модели (только она и дана)

```text
  context (B, 2W)
        │
        ▼
  ┌────────────────────────────┐
  │  Embedding(V, d)           │   входная матрица: слово как КОНТЕКСТ
  │  E_in ∈ R^{V × d}          │
  └────────────────────────────┘
        │  (B, 2W, d)
        ▼
  ┌────────────────────────────┐
  │  mean по оси 1 (соседям)    │   h = среднее векторов контекста
  └────────────────────────────┘
        │  h (B, d)
        ▼
  ┌────────────────────────────┐
  │  Linear(d → V, bias=False) │   выходная матрица: слово как ЦЕНТР
  │  E_out ∈ R^{V × d}         │   логит слова c:  E_out[c] · h
  └────────────────────────────┘
        │  logits (B, V)
        ▼
     softmax  →  p(центр | контекст)      (в обучении: CrossEntropyLoss)
```

**Требование к интерфейсу** (на него опираются готовые проверки в части 5):
у модели обязано быть поле `self.embeddings` (`nn.Embedding`) — это и есть
искомые векторы слов.

### ✏️ Задание 3 — модель

Реализуйте `CBOW` строго по схеме: две матрицы (входная `self.embeddings`,
выходная `self.output`), усреднение контекста, проекция в логиты.

**Напишите решение в ячейке ниже.**

In [ ]:
# ✏️ Задание 3 — модель CBOW (по схеме выше)
EMBED_DIM = 100

class CBOW(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        # TODO: self.embeddings = nn.Embedding(vocab_size, embed_dim)
        # TODO: self.output     = nn.Linear(embed_dim, vocab_size, bias=False)
        ...

    def forward(self, context):
        '''context: (B, 2W) -> logits: (B, V).'''
        # TODO: усреднить эмбеддинги контекста и спроецировать в логиты
        ...


# ---------- проверки (Задание 3) ----------
model = CBOW(V, EMBED_DIM).to(device)

logits = model(ctx_batch.to(device))
assert logits.shape == (ctx_batch.shape[0], V), f'форма логитов {tuple(logits.shape)}'
assert isinstance(model.embeddings, nn.Embedding), 'нужно поле self.embeddings (nn.Embedding)'

n_params = sum(p.numel() for p in model.parameters())
print('Задание 3 OK ✓ | logits', tuple(logits.shape), '| параметров:', f'{n_params:,}')

---

# Часть 4. Обучение

### ✏️ Задание 4 — один шаг обучения

Классические шаги: обнулить градиенты → forward → loss → `backward()` → `step()`.
Функция получает уже готовые `optimizer` и `criterion` (`nn.CrossEntropyLoss`).
Верните числовое значение loss (`loss.item()`).

**Напишите решение в ячейке ниже.**

In [ ]:
# ✏️ Задание 4 — шаг обучения
def train_step(model, context, target, optimizer, criterion):
    '''Один шаг обучения по батчу. Возвращает loss (float).'''
    model.train()
    context, target = context.to(device), target.to(device)
    # <-- YOUR CODE HERE: zero_grad -> forward -> loss -> backward -> step
    ...


# ---------- проверка: loss должен падать на одном и том же батче ----------
_o = torch.optim.Adam(model.parameters(), lr=1e-2)
_crit = nn.CrossEntropyLoss()
_c, _t = ctx_batch.to(device), tgt_batch.to(device)
with torch.no_grad():
    before = float(_crit(model(_c), _t))
for _ in range(20):
    after = train_step(model, _c, _t, _o, _crit)
assert after < before, 'loss не уменьшился — проверьте zero_grad/backward/step'
print('Задание 4 OK ✓ | loss до', round(before, 4), '-> после 20 шагов', round(after, 4))

### ✏️ Задание 5 — цикл обучения с валидацией

Для каждой эпохи:

1. **train**: один проход по `train_loader`, средний loss складываем в
   `history['train']` (удобно использовать `train_step`);
2. **валидация**: проход по `val_loader` **без градиентов**
   (`model.eval()` + `torch.no_grad()`), средний loss — в `history['val']`.

В конце верните `history` — словарь `{'train': [...], 'val': [...]}`.

**Напишите решение в ячейке ниже.**

In [ ]:
# ✏️ Задание 5 — цикл обучения
EPOCHS = 6
LR = 3e-3

def train_cbow(model, train_loader, val_loader, epochs, lr, verbose=True):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = {'train': [], 'val': []}
    for epoch in range(epochs):
        # ---- train ----
        model.train()
        # <-- YOUR CODE HERE: пройти train_loader, накопить средний loss

        # ---- validation ----
        model.eval()
        # <-- YOUR CODE HERE: пройти val_loader под torch.no_grad()

        if verbose:
            print(f'эпоха {epoch+1}/{epochs} | train {history["train"][-1]:.4f} '
                  f'| val {history["val"][-1]:.4f}')
    return history


# ---------- обучение ----------
torch.manual_seed(SEED)
model = CBOW(V, EMBED_DIM).to(device)     # чистая модель
t0 = time.time()
history = train_cbow(model, train_loader, val_loader, EPOCHS, LR)
print(f'время обучения: {time.time() - t0:.1f} c')

assert len(history['train']) == EPOCHS and len(history['val']) == EPOCHS
assert history['train'][-1] < history['train'][0], 'train loss должен падать'
print('Задание 5 OK ✓ | train loss', round(history['train'][0], 3),
      '->', round(history['train'][-1], 3))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(range(1, EPOCHS + 1), history['train'], 'o-', color='tomato', label='train')
ax.plot(range(1, EPOCHS + 1), history['val'],   's--', color='steelblue', label='val')
ax.set_xlabel('эпоха')
ax.set_ylabel('CrossEntropy loss (полный softmax)')
ax.set_title('CBOW: loss на train и валидации')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

---

# Часть 5. Проверки близости слов (готовое решение)

Эта часть — **готовый код**, писать не нужно. Он берёт входную матрицу
`model.embeddings`, нормирует её по L2 и ищет ближайшие слова по косинусу. Именно
ради этой проверки модель обязана иметь поле `self.embeddings`.

In [ ]:
@torch.no_grad()
def get_embeddings_matrix(model):
    '''L2-нормированная матрица входных эмбеддингов (V, d).'''
    E = model.embeddings.weight.detach()
    return E / (E.norm(dim=1, keepdim=True) + 1e-9)

@torch.no_grad()
def nearest_words(word, k=6, En=None):
    '''k ближайших слов по косинусу (само слово исключаем).'''
    if word not in word2id:
        return []
    if En is None:
        En = get_embeddings_matrix(model)
    sims = En @ En[word2id[word]]
    order = sims.argsort(descending=True).tolist()
    return [id2word[j] for j in order if j != word2id[word]][:k]

In [ ]:
En = get_embeddings_matrix(model)

print('Ближайшие соседи (CBOW, полный softmax):\n')
for w in ['king', 'city', 'war', 'music', 'water', 'france',
          'school', 'president', 'river', 'energy', 'dog', 'film']:
    if w in word2id:
        print(f'  {w:11s} -> ' + ', '.join(nearest_words(w, k=6, En=En)))

## 5.1. Линейные аналогии (готово)

Классический тест word2vec — векторная арифметика: `france − paris + italy ≈ rome`
(«страна − столица + столица ≈ страна»). Векторы складывают и вычитают, затем ищут
ближайшее слово-кандидат (саму тройку из запроса исключаем). На небольшом корпусе
аналогии часто «грязные» — это нормально.

In [ ]:
@torch.no_grad()
def analogy(a, b, c, k=5):
    '''a − b + c ≈ ?  (слова-токены)'''
    if any(w not in word2id for w in (a, b, c)):
        return []
    En = get_embeddings_matrix(model)
    vec = En[word2id[a]] - En[word2id[b]] + En[word2id[c]]
    vec = torch.nn.functional.normalize(vec, dim=0)
    sims = En @ vec
    order = sims.argsort(descending=True).tolist()
    banned = {word2id[a], word2id[b], word2id[c]}
    return [id2word[j] for j in order if j not in banned][:k]

for a, b, c in [('france', 'paris', 'italy'),
                ('france', 'paris', 'germany'),
                ('king', 'queen', 'man')]:
    print(f'{a} - {b} + {c} -> ' + ', '.join(analogy(a, b, c)))

---

## 6. Сохранение эмбеддингов (готово)

Сохраним матрицу эмбеддингов, словарь и гиперпараметры — пригодится дальше.
Формат `.npz` самодостаточен.

In [ ]:
OUT_PATH = 'cbow_embeddings.npz'

np.savez(
    OUT_PATH,
    embeddings=get_embeddings_matrix(model).cpu().numpy(),   # нормированные (для косинусов)
    raw_embeddings=model.embeddings.weight.detach().cpu().numpy(),
    vocab=np.array(id2word, dtype=object),
    embed_dim=EMBED_DIM,
    window=WINDOW,
)
print('сохранено в', OUT_PATH)

loaded = np.load(OUT_PATH, allow_pickle=True)
print('проверка: embeddings', loaded['embeddings'].shape, '| словарь', loaded['vocab'].shape)

---

## 7. Итоги и чеклист

**Что мы сделали.** Прочитали текст, построили словарь, собрали пары
*(контекст → центр)* скользящим окном, написали `Dataset`, модель CBOW с полным
`softmax` и цикл обучения — и получили плотные векторы, где близость по смыслу
превращается в близость по косинусу.

**Ключевые идеи:**

| Приём | Зачем |
|---|---|
| Скользящее окно | из неразмеченного текста получаем обучающие пары |
| Усреднение контекста | CBOW дёшев: один вектор $h$ на пример |
| Полный `softmax` | предсказываем центр среди всех слов (без negative sampling) |
| `CrossEntropyLoss` | стандартная функция потерь для $V$-классовой задачи |
| Две матрицы ($E_{in}, E_{out}$) | слово отдельно как контекст и как центр |

**Чеклист сдачи:**

- [ ] Задание 1 — `make_windows` (окна контекста);
- [ ] Задание 2 — `CBOWDataset`;
- [ ] Задание 3 — модель `CBOW`;
- [ ] Задание 4 — `train_step`;
- [ ] Задание 5 — `train_cbow` (цикл с валидацией);
- [ ] «Задание N OK ✓» по всем пяти заданиям;
- [ ] график loss, блок соседей и аналогий, сохранённые эмбеддинги.

> **Резюме одной строкой.** CBOW = усреднить контекст → предсказать центр;
> полный `softmax` делает обучение «честным» $V$-классовым, и на выходе
> получаются плотные векторы, где близость по смыслу = близость по косинусу.

---

**Полезные ссылки**

- [Efficient Estimation of Word Representations in Vector Space (Mikolov et al., 2013)](https://arxiv.org/abs/1301.3781) — оригинальный word2vec (CBOW и Skip-gram).
- [word2vec Parameter Learning Explained (Rong, 2014)](https://arxiv.org/abs/1411.2738) — подробный вывод градиентов CBOW.